In [1]:
import requests
import json


data = {
    "query": "什么是氩弧焊",
    "mode": "local_kb",
    "kb_name": "电焊demo",
    "top_k": 3,
    "score_threshold": 2,
    "history": [
        {
            "content": "你好，请问你了解电焊知识吗",
            "role": "user"
        },
        {
            "content": "我是电焊领域的专家，很乐意为你解答！",
            "role": "assistant"
        }
    ],
    "stream": False,
    "temperature": 0.7,
    "max_tokens": 1024,
    "prompt_name": "default",
    "return_direct": False
}

# 假设请求的URL
url = "http://202.38.247.12:7861/chat/kb_chat"  # 请替换为实际的API URL


# 发送POST请求
response = requests.post(url, json=data)
response.raise_for_status()  # 检查请求是否成功

# 解析响应的JSON数据
response_json = response.json()
 


In [5]:
response_json = json.loads(response.json())
choice = response_json["choices"][0]

In [6]:
choice

{'message': {'role': 'assistant',
  'content': '氩弧焊是一种使用氩气保护焊接过程，在焊接时，钨极与工作件之间形成的电弧在氩气保护下进行焊接，通过产生的高温使金属融化并形成焊接。',
  'finish_reason': None,
  'tool_calls': []}}

In [5]:
from app.utils.paramSQL import access_DB

def get_param_by_mat_met_thi(MAT, MET, THI):
    # （一）拼接SQL语句
    query = f"""
        SELECT Diameter, ParamName, ParamValue
        FROM param_data
        WHERE Method = '{MET}'
        AND Material = '{MAT}'
        AND Thickness = '{THI}'
        """

    # （二）连接数据库并执行查询
    result = access_DB(query)

    # （三）返回结果
    return result

In [6]:
results = get_param_by_mat_met_thi("Fe", "MIG", "5")

In [7]:
results

[('0.8', 'AUTOSetSpeed', '181'),
 ('0.8', 'CraterOffTime', '20'),
 ('0.8', 'CraterTime', '250'),
 ('0.8', 'DAC_PreinstallBase_MIN', '2700'),
 ('0.8', 'DAC_PreinstallPeak_MAX', '36975'),
 ('0.8', 'Diameter', '0.8'),
 ('0.8', 'NoPulseInductanceRatioUp', '35000'),
 ('0.8', 'NoPulseInductanceRatioDown', '15000'),
 ('0.8', 'PluseArcMinCurrent', '2000'),
 ('0.8', 'PluseArcTime', '40'),
 ('0.8', 'PluseCycle', '4000'),
 ('0.8', 'PlusePeakAverageCurrent', '20'),
 ('0.8', 'PlusePeakMaxCurrent', '325'),
 ('0.8', 'PlusePeakTime', '21'),
 ('0.8', 'PlusePeakVoltage', '490'),
 ('0.8', 'PulseInductanceRatioUp', '1200'),
 ('0.8', 'PulseInductanceRatioDown', '4000'),
 ('0.8', 'PulseSetCurrent', '20'),
 ('0.8', 'PulseSetVoltage', '135'),
 ('0.8', 'SetCurrent', '20'),
 ('0.8', 'SetPID_Ki', '9'),
 ('0.8', 'SetPID_Kp', '8000'),
 ('0.8', 'SetVoltage', '135'),
 ('0.8', 'SetVoltage_ArCO2', '120'),
 ('0.8', 'Speed', '0'),
 ('0.8', 'StartCurrentTime', '250')]

In [ ]:
results = get_param_by_mat_met_thi("Fe", "MIG", "5")
data = [{f"{item[1]}": f"{item[2]}"} for item in results]

In [1]:
def param_query(param_dict: dict):
    try:
        MAT = param_dict['MAT']
        MET = param_dict['MET']
        THI = param_dict['THI']
        # DIA = param_dict['DIA']
    except:
        return {
            "response": "参数查询失败，请检查输入的参数格式是否正确。"
        }
    from app.utils.paramSQL import get_param_by_mat_met_thi
    results = get_param_by_mat_met_thi(MAT, MET, THI)
    if not results or len(results) == 0:
        return {
            "response": "未找到相关参数信息。"
        }
    if results[0][0] == "0":
        ## 没有焊丝直径的参数
        data = [{f"{item[1]}": f"{item[2]}"} for item in results]
        return {
            "response": "查询成功，以下是相关参数信息：",
            "data": data
        }
    else:
        ## 有焊丝直径的参数
        try:
            DIA = param_dict['DIA']
        except:
            return {
                "response": "参数查询失败，请检查输入的参数格式是否正确。"
            }
        results_this_DIA = [item for item in results if item[0] == str(DIA)]
        if not results_this_DIA or len(results_this_DIA) == 0:
            return {
                "response": "未找到该焊丝直径的相关参数信息。"
            }
        data = [{f"{item[1]}": f"{item[2]}"} for item in results_this_DIA]
        return {
            "response": "查询成功，以下是相关参数信息：",
            "data": data
        }

In [3]:
param_dict = {
    "MAT": "Fe",
    "MET": "MIG",
    "THI": "5",
    "DIA": "0.8"
}
param_query(param_dict)

{'response': '查询成功，以下是相关参数信息：',
 'data': [{'AUTOSetSpeed': '181'},
  {'CraterOffTime': '20'},
  {'CraterTime': '250'},
  {'DAC_PreinstallBase_MIN': '2700'},
  {'DAC_PreinstallPeak_MAX': '36975'},
  {'Diameter': '0.8'},
  {'NoPulseInductanceRatioUp': '35000'},
  {'NoPulseInductanceRatioDown': '15000'},
  {'PluseArcMinCurrent': '2000'},
  {'PluseArcTime': '40'},
  {'PluseCycle': '4000'},
  {'PlusePeakAverageCurrent': '20'},
  {'PlusePeakMaxCurrent': '325'},
  {'PlusePeakTime': '21'},
  {'PlusePeakVoltage': '490'},
  {'PulseInductanceRatioUp': '1200'},
  {'PulseInductanceRatioDown': '4000'},
  {'PulseSetCurrent': '20'},
  {'PulseSetVoltage': '135'},
  {'SetCurrent': '20'},
  {'SetPID_Ki': '9'},
  {'SetPID_Kp': '8000'},
  {'SetVoltage': '135'},
  {'SetVoltage_ArCO2': '120'},
  {'Speed': '0'},
  {'StartCurrentTime': '250'}]}

In [26]:
results = get_param_by_mat_met_thi("Fe", "MIG", "5")

In [27]:
results

[('0.8', 'AUTOSetSpeed', '181'),
 ('0.8', 'CraterOffTime', '20'),
 ('0.8', 'CraterTime', '250'),
 ('0.8', 'DAC_PreinstallBase_MIN', '2700'),
 ('0.8', 'DAC_PreinstallPeak_MAX', '36975'),
 ('0.8', 'Diameter', '0.8'),
 ('0.8', 'NoPulseInductanceRatioUp', '35000'),
 ('0.8', 'NoPulseInductanceRatioDown', '15000'),
 ('0.8', 'PluseArcMinCurrent', '2000'),
 ('0.8', 'PluseArcTime', '40'),
 ('0.8', 'PluseCycle', '4000'),
 ('0.8', 'PlusePeakAverageCurrent', '20'),
 ('0.8', 'PlusePeakMaxCurrent', '325'),
 ('0.8', 'PlusePeakTime', '21'),
 ('0.8', 'PlusePeakVoltage', '490'),
 ('0.8', 'PulseInductanceRatioUp', '1200'),
 ('0.8', 'PulseInductanceRatioDown', '4000'),
 ('0.8', 'PulseSetCurrent', '20'),
 ('0.8', 'PulseSetVoltage', '135'),
 ('0.8', 'SetCurrent', '20'),
 ('0.8', 'SetPID_Ki', '9'),
 ('0.8', 'SetPID_Kp', '8000'),
 ('0.8', 'SetVoltage', '135'),
 ('0.8', 'SetVoltage_ArCO2', '120'),
 ('0.8', 'Speed', '0'),
 ('0.8', 'StartCurrentTime', '250')]

## 单元测试：Integrate 推荐逻辑 到华数接口

In [2]:
from app.utils.paramSQL import *

all_params = get_all_param_by_MAT_MET('MIG', 'AlMg')

In [5]:
processed_params_dict = {}  # 目标格式：{DIA: {paramName: [(THI, paramValue)]}}
# [('0', '0.8', 'AUTOSetSpeed', '20'),
#  ('0', '1', 'AUTOSetSpeed', '20'),
#  ('0', '0.8', 'CraterOffTime', '0')]
for thi, dia, name, value in all_params:
    if dia not in processed_params_dict:
        processed_params_dict[dia] = {}
    if name not in processed_params_dict[dia]:
        processed_params_dict[dia][name] = []
    processed_params_dict[dia][name].append((thi, value))

In [1]:
from app.services.bert_param_recommend import thickness_recommend

thickness_recommend('AlMg', 'MIG', '5')

/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'0.8': {'AUTOSetSpeed': 180.0,
  'CraterOffTime': 20.0,
  'CraterTime': 100.0,
  'DAC_PreinstallBase_MIN': 3200.0,
  'DAC_PreinstallPeak_MAX': 20000.0,
  'Diameter': 0.8,
  'NoPulseInductanceRatioUp': 0.0,
  'NoPulseInductanceRatioDown': 0.0,
  'PluseArcMinCurrent': 3200.0,
  'PluseArcTime': 20.0,
  'PluseCycle': 3600.0,
  'PlusePeakAverageCurrent': 15.0,
  'PlusePeakMaxCurrent': 280.0,
  'PlusePeakTime': 20.0,
  'PlusePeakVoltage': 340.0,
  'PulseInductanceRatioUp': 3000.0,
  'PulseInductanceRatioDown': 3000.0,
  'PulseSetCurrent': 20.0,
  'PulseSetVoltage': 100.0,
  'SetCurrent': 20.0,
  'SetPID_Ki': 9.0,
  'SetPID_Kp': 8000.0,
  'SetVoltage': 100.0,
  'SetVoltage_ArCO2': 100.0,
  'Speed': 0.0,
  'StartCurrentTime': 250.0},
 '1': {'AUTOSetSpeed': 125.0,
  'CraterOffTime': 0.0,
  'CraterTime': 0.0,
  'DAC_PreinstallBase_MIN': 0.0,
  'DAC_PreinstallPeak_MAX': 0.0,
  'Diameter': 1.0,
  'NoPulseInductanceRatioUp': 0.0,
  'NoPulseInductanceRatioDown': 0.0,
  'PluseArcMinCurrent': 0.0,
  

In [2]:
from app.api.endpoints.session import param_query

param_query({'MAT': 'AlMg', 'MET': 'MIG', 'THI': '2.3', 'DIA': "1.6"})

/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/home/zkyao/miniconda3/envs/weld_gpt/lib/python3.8/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please u

SELECT Thickness FROM param_data WHERE Method = 'MIG' AND Material = 'AlMg';
[]
[('0.8', 'AUTOSetSpeed', 95.5), ('0.8', 'CraterOffTime', 0.0), ('0.8', 'CraterTime', 0.0), ('0.8', 'DAC_PreinstallBase_MIN', 0.0), ('0.8', 'DAC_PreinstallPeak_MAX', 0.0), ('0.8', 'Diameter', 0.8), ('0.8', 'NoPulseInductanceRatioUp', 0.0), ('0.8', 'NoPulseInductanceRatioDown', 0.0), ('0.8', 'PluseArcMinCurrent', 0.0), ('0.8', 'PluseArcTime', 0.0), ('0.8', 'PluseCycle', 0.0), ('0.8', 'PlusePeakAverageCurrent', 0.0), ('0.8', 'PlusePeakMaxCurrent', 0.0), ('0.8', 'PlusePeakTime', 0.0), ('0.8', 'PlusePeakVoltage', 0.0), ('0.8', 'PulseInductanceRatioUp', 0.0), ('0.8', 'PulseInductanceRatioDown', 0.0), ('0.8', 'PulseSetCurrent', 0.0), ('0.8', 'PulseSetVoltage', 0.0), ('0.8', 'SetCurrent', 0.0), ('0.8', 'SetPID_Ki', 0.0), ('0.8', 'SetPID_Kp', 0.0), ('0.8', 'SetVoltage', 0.0), ('0.8', 'SetVoltage_ArCO2', 0.0), ('0.8', 'Speed', 0.0), ('0.8', 'StartCurrentTime', 0.0), ('1', 'AUTOSetSpeed', 66.0), ('1', 'CraterOffTime',

{'response': '查询成功，以下是相关参数信息：',
 'data': [{'AUTOSetSpeed': '39.3'},
  {'CraterOffTime': '40.0'},
  {'CraterTime': '150.0'},
  {'DAC_PreinstallBase_MIN': '2111.44'},
  {'DAC_PreinstallPeak_MAX': '17964.27'},
  {'Diameter': '1.6'},
  {'NoPulseInductanceRatioUp': '0.0'},
  {'NoPulseInductanceRatioDown': '0.0'},
  {'PluseArcMinCurrent': '1820.0'},
  {'PluseArcTime': '50.0'},
  {'PluseCycle': '631.52'},
  {'PlusePeakAverageCurrent': '25.0'},
  {'PlusePeakMaxCurrent': '340.0'},
  {'PlusePeakTime': '36.0'},
  {'PlusePeakVoltage': '380.0'},
  {'PulseInductanceRatioUp': '1000.0'},
  {'PulseInductanceRatioDown': '3027.82'},
  {'PulseSetCurrent': '169.65'},
  {'PulseSetVoltage': '172.84'},
  {'SetCurrent': '169.65'},
  {'SetPID_Ki': '2.34'},
  {'SetPID_Kp': '4000.0'},
  {'SetVoltage': '169.65'},
  {'SetVoltage_ArCO2': '169.65'},
  {'Speed': '2.05'},
  {'StartCurrentTime': '550.0'}]}

In [ ]:
from app.services.bert_param_recommend import thickness_recommend

recommend_results = thickness_recommend('AlMg', 'MIG', '5')
results = []
for DIA, params_this_DIA in recommend_results.items():
    for ParamName, ParamValue in params_this_DIA.items():
        results.append((DIA, ParamName, ParamValue))

In [6]:
results_this_DIA = [item for item in results if item[0] == str(DIA)]
results_this_DIA

[('1.6', 'AUTOSetSpeed', 38.68),
 ('1.6', 'CraterOffTime', 40.0),
 ('1.6', 'CraterTime', 150.0),
 ('1.6', 'DAC_PreinstallBase_MIN', 2153.93),
 ('1.6', 'DAC_PreinstallPeak_MAX', 18027.29),
 ('1.6', 'Diameter', 1.6),
 ('1.6', 'NoPulseInductanceRatioUp', 0.0),
 ('1.6', 'NoPulseInductanceRatioDown', 0.0),
 ('1.6', 'PluseArcMinCurrent', 1820.0),
 ('1.6', 'PluseArcTime', 50.0),
 ('1.6', 'PluseCycle', 622.94),
 ('1.6', 'PlusePeakAverageCurrent', 25.0),
 ('1.6', 'PlusePeakMaxCurrent', 340.0),
 ('1.6', 'PlusePeakTime', 36.0),
 ('1.6', 'PlusePeakVoltage', 380.0),
 ('1.6', 'PulseInductanceRatioUp', 1000.0),
 ('1.6', 'PulseInductanceRatioDown', 2996.29),
 ('1.6', 'PulseSetCurrent', 171.66),
 ('1.6', 'PulseSetVoltage', 174.8),
 ('1.6', 'SetCurrent', 171.66),
 ('1.6', 'SetPID_Ki', 2.32),
 ('1.6', 'SetPID_Kp', 4000.0),
 ('1.6', 'SetVoltage', 171.66),
 ('1.6', 'SetVoltage_ArCO2', 171.66),
 ('1.6', 'Speed', 2.17),
 ('1.6', 'StartCurrentTime', 550.0)]

In [1]:
from app.utils.paramSQL import get_all_THI
all_THI = get_all_THI(MET="MIG", MAT="AlMg")

SELECT Thickness FROM param_data WHERE Method = 'MIG' AND Material = 'AlMg';


In [3]:
all_THI

[]